# 深度學習與金融應用｜Unit 4
# RNN / LSTM 時序模型

**教師版(Colab 版,含解答與藍框註解)**

> 本版本以 **Google Colab** 為環境,學生教學主線。本單元運算量較大,請先切到 GPU:**執行階段 → 變更執行階段類型 → T4 GPU**。

---

**學習目標**
1. 理解序列資料為什麼不能用一般 MLP,順序資訊為何重要
2. 理解 RNN 的核心機制(hidden state 邊讀邊記),並**親手實作 SimpleRNN**
3. 用「記憶力測試」實驗**親眼看到梯度消失**:SimpleRNN 為什麼記不久(概念,不推導)
4. 理解 LSTM / GRU 的閘門直覺,以及閘門如何解決「記不久」的問題
5. 會用滑動視窗製作時序監督式樣本,並理解**預測距離(horizon)**如何改變題目難度與 baseline 強弱
6. 實作 **walk-forward 驗證**(時序資料的正確驗證方式)
7. **真實資料的正面成果**:氣溫預測中,LSTM 的誤差比 naive baseline 低約五成
8. **問題怎麼問決定勝負**:同樣是波動率,預測「數值」贏不過 naive,改預測「變化方向」才贏得了

**氣象案例(主線正面成果)**:德國 Jena 氣象站**氣溫**預測

**金融案例(合成資料)**:模擬台股報酬率的**波動率**預測(不是報酬率方向!)

> 第 5~7 節使用**合成**報酬率(內建波動率群聚與均值回歸),目的是乾淨示範「結構存在時模型抓不抓得到」。真實 2330 資料上的結論請見 U1。

**預計時數**:10 小時


---
## 第 0 節　環境確認(Colab)


In [ ]:
import tensorflow as tf
print("TensorFlow 版本:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("✅ GPU 已啟用" if gpus else "ℹ️ 目前為 CPU(記得切換執行階段類型)")


---
## 第 1 節　為什麼序列資料不能用一般 MLP?

先問一個問題:如果把「過去 5 天的收盤價」丟給 U1 那種一般的神經網路(MLP),它會怎麼看這 5 個數字?

答案是:它把它們當成 <b>5 個彼此無關的特徵</b>,就像「身高、體重、年齡、收入、血壓」那樣並排放著,<b>誰先誰後,它完全沒感覺</b>。

舉個例子就很清楚。假設兩檔股票這 5 天的收盤價是:

- A:`100 → 102 → 104 → 106 → 108`(一路上漲)
- B:`108 → 106 → 104 → 102 → 100`(一路下跌)

你我一眼就知道這是<b>完全相反</b>的兩件事。但對 MLP 來說,這只是同一組數字換了擺放位置而已。它並不知道左邊那格叫「5 天前」、右邊那格叫「昨天」,更不知道這兩格之間有「先後」關係。順序這件事,它只能從大量資料裡自己硬學,而不是一開始就內建在結構裡。

這就是核心:<b>時間序列裡,「順序」本身就是資訊。</b>就像「狗咬人」和「人咬狗」用的是同樣三個字,一個是日常、一個是新聞。

除此之外,MLP 用在序列上還有兩個實際的不方便:

1. <b>輸入長度被寫死</b>:你決定「用過去 5 天」,模型就只能吃 5 天。想改成看 20 天,得整個重蓋。
2. <b>學到的東西不能共用</b>:「連續上漲」這個型態出現在第 1~3 天、和出現在第 3~5 天,對 MLP 而言是兩件要分開學的事,得各學一次,很浪費。

### RNN:一邊讀、一邊記

<b>RNN(循環神經網路,Recurrent Neural Network)就是為了這些問題而生的。</b>它不把整段序列一次攤平來看,而是<b>一個時間點、一個時間點地讀進去</b>,同時在「心裡」維護一份記憶,術語叫 <b>hidden state(隱藏狀態)</b>。

用讀小說來比喻。你讀到第 200 頁時,不會把前 199 頁的每個字都背在腦中;你腦中有的是一份<b>劇情摘要</b>:誰是主角、誰跟誰結了仇、上一章發生了什麼。每讀一頁,你就用「舊摘要 + 這一頁的新內容」更新出一份新摘要。RNN 做的事一模一樣:

<p style="text-align:center;"><b>新記憶 = f(舊記憶, 新輸入)</b></p>

每讀一個時間點就套用一次這條規則,所以:

- <b>順序天生被保留</b>:先讀的東西先進入記憶,後讀的東西是在「已有記憶」的基礎上被理解的。「一路上漲」和「一路下跌」讀完後,留下的記憶完全不同。
- <b>序列多長都吃得下</b>:因為每一步用的都是<b>同一套權重</b>(同一條更新規則),看 5 天和看 60 天,用的是同一顆模型。
- <b>學到的型態可以共用</b>:「連續上漲」不管出現在序列的哪一段,經過的都是同一條更新規則,學一次就到處適用。

### 但陽春 RNN 有個致命弱點:記不久

聽起來很完美,不過最陽春的 RNN(Keras 裡叫 <code>SimpleRNN</code>)有一個致命傷。

訓練神經網路靠的是把「預測誤差」往回傳、據此修正權重(U1 講過的反向傳播)。對 RNN 來說,誤差要<b>沿著時間軸一路往回傳</b>:從最後一步傳回倒數第二步、再往前、一直傳回第一步。問題是這個訊號<b>每傳一步就衰減一次</b>,像傳話遊戲:傳到第一百個人時,原話早就沒了。

結果就是:序列一長,開頭那幾步幾乎收不到任何學習訊號,模型<b>記不住很久以前的事</b>。這個現象有個正式名字,叫<b>梯度消失(vanishing gradient)</b>。名字先記著就好,本課不推導數學,因為等一下我們會做一個實驗,讓你<b>親眼看到</b>它發生。

本單元的路線是:第 2 節先讓 SimpleRNN 在短序列上成功一次(證明它真的能學序列,工具也沒問題);第 2.5 節把序列拉長,看著 SimpleRNN 撐不住而失敗;然後才輪到 LSTM 登場,並把它加回同一個測試,你就會親眼看到它的閘門設計解決了眼前哪個具體的問題。


---
## 第 2 節　暖身:用 SimpleRNN 預測正弦波

在拿循環網路去碰金融資料之前,我們先做一件看起來有點「作弊」的事:用<b>最陽春的 RNN</b>(Keras 的 <code>SimpleRNN</code>)去預測一條<b>正弦波</b>。

為什麼要花時間在這種一看就有規律的假資料上?因為它能幫我們把兩種失敗<b>分開來</b>:

- <b>模型或程式有問題</b>:資料明明有規律,模型卻學不起來(shape 弄錯、忘了標準化、層數接錯……)
- <b>資料本身沒訊號</b>:模型和程式都對,但資料裡根本沒有東西可學

這兩種狀況在畫面上都長成「預測不準」,原因卻天差地遠。而正弦波是一個<b>確定有答案</b>的題目:如果連它都學不起來,那一定是我們自己寫錯了,不必懷疑資料。

先把工具在「已知有解」的題目上驗證過,之後遇到金融資料預測不準,才有底氣說:<b>這次是資料的問題,不是我的問題。</b>

這一節你會看到三件事:怎麼把一條連續訊號切成模型吃得下的樣本、一個 SimpleRNN 模型長什麼樣、以及一張「預測 vs 實際」幾乎重疊的圖。看到那張圖,代表兩件事同時成立:工具沒問題,而且<b>陽春 RNN 在短序列上是夠用的</b>。它的極限在哪裡,是下一節(2.5)的事。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# np.linespace()主要就是在區間內等距取2000個數值，數值介於0~100之間
t = np.linspace(0, 100, 2000)
t

# np.sin():對每個數值取正弦值
# 0.1 * np.random.randn(2000)，目的就是加一點雜訊值在正弦值內，加上2000個雜訊值
# randn()是用來產生標準常態分布的隨機數，* 0.1的目的是為了把振幅壓小
# 產生有規律(正弦)但又不完美(有雜訊)的序列，長度是2000
sine = np.sin(t) + 0.1 * np.random.randn(2000)
sine

# 畫出前200個點
plt.figure(figsize=(12, 3))
plt.plot(t[:200], sine[:200])
plt.title('Sine Wave Singal')
plt.xlabel('t')
plt.show()

### 滑動視窗:把序列變成監督式學習樣本

時序預測的關鍵技巧:用「過去 N 個點」當輸入 X,「下一個點」當答案 y。

問題是,原始的正弦波只是**一長條 2000 個數字**,並沒有現成的「輸入 / 答案」配對。監督式學習需要一堆 (X, y) 樣本才能訓練,所以我們得先自己「切」出來——這就是**滑動視窗(sliding window)**。

做法像拿一個固定寬度的框,在序列上**一格一格往右滑**:框裡的 N 個點當成一筆輸入 X,框「右邊緊鄰的下一個點」當成答案 y;滑一步就產生一筆樣本。以 `WIN=3` 的迷你例子說明,序列 `[10, 11, 12, 13, 14, 15]` 會被切成:

| 樣本 | 輸入 X(過去 3 個) | 答案 y(下一個) |
|---|---|---|
| 1 | `[10, 11, 12]` | `13` |
| 2 | `[11, 12, 13]` | `14` |
| 3 | `[12, 13, 14]` | `15` |

相鄰兩筆樣本**重疊了大部分內容、只往前推一步**,所以一條長度 L 的序列能切出約 `L − WIN` 筆樣本,資料量一下子變多。下面的 `make_windows` 就是把這個「切框」動作寫成迴圈。

三個實作上要留意的細節,對應下一格程式:

1. **視窗長度 `WIN` 是超參數**:一次看幾步由你決定(這裡設 20)。看太少會漏掉長期趨勢,看太多則每筆樣本變重、可用樣本數變少——這是要權衡的。
2. **要補上「特徵維」湊成三維**:Keras 的循環層(SimpleRNN/LSTM)吃的輸入形狀固定是 `(樣本數, 時間步, 特徵數)`。我們每個時間點只有一個數值,所以特徵數 = 1,用 `X[..., None]` 在最後補一個維度,把 `(樣本, 20)` 變成 `(樣本, 20, 1)`。
3. **切訓練/測試時「不能打亂順序」**:一般分類題會 shuffle 後隨機切,但時序資料若打亂,等於讓模型「用未來的資料去預測過去」,屬於資料洩漏。正確做法是**照時間先後切**——前 80% 當訓練、後 20% 當測試,確保測試集永遠在訓練集「之後」。

> 補充:這裡答案 y 取的是「**下一個**點」,也就是預測距離(horizon)= 1。之後在第 3、5 節你會把 horizon 拉大(例如預測 12 小時後、或未來 5 日平均),題目難度與 baseline 強弱都會跟著改變——但「用滑動視窗造樣本」這個骨架完全一樣。


In [ ]:
# 使用滑動視窗的方式切出正弦波的X與y

def make_window(series, win):
  X, y = [], [] # X代表要收集每一筆輸入視窗，y收集對應的答案

  # i 從win開始
  for i in range(win, len(series)):
    X.append(series[i-win:i]) # 取i之前的win個點
    y.append(series[i]) # 取第i個答案
  # 迴圈跑完會得到len(series)-win樣本，再轉成numpy陣列
  return np.array(X), np.array(y)

WIN = 20
X_sine, y_sine = make_window(sine, WIN)
X_sine.shape
y_sine.shape

# 因為RNN要求輸入需要是3維陣列，也就是說要樣本數、時間窗格、特徵數
# 因為我們每個時間點只有1個數值，所以特徵數=1，用[..., None]來補最後的維度
# [..., None]，...代表前面所有維度原封不動，None等同新增1個維度
X_sine = X_sine[..., None] # 新增1個維度
print(f'X shape: {X_sine.shape}, 樣本數, 時間窗格, 特徵數')

# 切訓練和測試資料集
# 時間序列的資料不能隨機打亂，因為隨機打亂會把訓練集資料混入測試集裡面，等於資料洩漏
# 照時間先後切出前80%資料進行訓練，後20%資料進行測試
sp = int(len(X_sine)*0.8) # 得出分割點
Xtr_s, Xte_s = X_sine[:sp], X_sine[sp:]
ytr_s, yte_s = y_sine[:sp], y_sine[sp:]
print(Xtr_s.shape)
print(Xte_s.shape)
print(ytr_s.shape)
print(yte_s.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

# 模型設置
sine_model = keras.Sequential([
        keras.Input(shape=(WIN, 1)),
        layers.SimpleRNN(32),
        layers.Dense(1)
])

# 模型編譯
sine_model.compile(optimizer='adam', loss='mse')

# 模型訓練
sine_model.fit(Xtr_s, ytr_s, epochs=10, batch_size=32, verbose=1)


In [ ]:
# 模型測試
pred_sine = sine_model.predict(Xte_s, verbose=0).ravel()
mse = np.mean((pred_sine - yte_s)**2)
print(f'正弦波預測MSE: {mse:.4f}(很小，但是RNN學得起來)')

# 畫出圖形
plt.figure(figsize=(12, 3))
plt.plot(yte_s[:], label='Actual')
plt.plot(pred_sine[:], label='Simple RNN Prediction', linestyle='--')
plt.title('Sine Wave: Actual V.S Predicted')
plt.legend()
plt.show()

---
## 記憶力測試:陽春 RNN 的極限

正弦波很成功,但別忘了那是<b>短視窗</b>:模型一次只需要看 20 步。現在來做一個殘忍一點的實驗,專門考「長程記憶」。

這個題目叫 <b>adding problem</b>(RNN 領域的經典測驗):給模型一條長度 T 的序列,每個時間步有<b>兩個數字</b>——一個是 0~1 之間的隨機值,另一個是<b>標記</b>(整條序列只有<b>兩個位置</b>是 1,其餘都是 0)。模型要回答的是:<b>那兩個被標記位置上的隨機值,加起來等於多少?</b>

為什麼這能考記憶?因為兩個標記可能一個在很前面、一個在很後面。模型必須在讀到第一個標記時<b>把那個數字記住</b>,一路撐過中間幾十上百步的干擾,直到讀到第二個標記,才能把兩者相加。<b>序列越長,要記得越久。</b>這正是閘門派得上用場、而陽春 RNN 會吃鱉的地方。

這一節先只考<b>陽春的 SimpleRNN</b>:讓它在長度 20、50、100 的序列上各考一次,看它的誤差(MSE,<b>越低越好</b>)隨長度怎麼變。對照基準是「不管三七二十一都猜 1.0」(兩個 0~1 隨機值的期望和),它的 MSE 約 0.17——模型要是掉到這條線,就等於什麼都沒學到。


In [ ]:
# 造adding problem的資料，它會回傳X:(n, T, 2)、y:(n, )
# n = 樣本數，T = 序列長度，每步2個特徵的序列
def make_adding(T, n=6000, seed=42):
  # rnag = 固定亂數種子，資料可重現
  rng = np.random.default_rng(seed)
  # vals = 每個時間步會有一個0~1的隨機數
  vals = rng.random((n, T, 1)).astype('float32')
  # marks = 從每條序列裡面挑兩個位置設成1，其餘為0
  marks = np.zeros((n, T, 1), dtype='float32')
  # 逐條序列處理
  for i in range(n):
    # j, k = 隨機挑兩個不重複的位置
    j, k = rng.choice(T, size=2, replace=False)
    # 把這兩個位置都標上1
    marks[i, j, 0] = 1.0
    marks[i, k, 0] = 1.0
  # 把val和marks沿著軸疊起來
  X = np.concatenate([vals, marks], axis=2)
  y = (vals[:, :, 0] * marks[:, :, 0]).sum(axis=1)
  return X, y

# 目的是訓練一個RNN來解adding problem，所以結果就是回傳測試集的MSE(越低越好)
def adding_test(T, cell_type, units=32, epochs=60, seed=42):
  X, y = make_adding(T, seed=seed)
  tf.random.set_seed(seed)
  m = keras.Sequential([
      keras.Input(shape=(T, 2)),
      cell_type(units),
      layers.Dense(1)
  ])
  m.compile(optimizer='adam', loss='mse')
  tr = int(len(X) * 0.8)
  m.fit(X[:tr], y[:tr], epochs=epochs, batch_size=64, verbose=1)
  return m.evaluate(X[tr:], y[tr:], verbose=0)

# Baseline:完全不看序列
# 它的MSE會是等於y的變異數，也等於2x(1/12) = 1/6，接近0.167
# 所以當模型的MSE掉到0.167以下，代表什麼都沒學
BASELINE = 1/6
length = [20, 50, 100] # 3種序列長度，越長的話代表第一個標記到第二個標記的距離可能很遠
mse_rnn = [] # 儲存RNN在各個序列下的MSE

for T in length:
  e = adding_test(T, layers.SimpleRNN)
  mse_rnn.append(e)
  print(f'序列長度: {T}:Simple RNN MSE: {e:.4f}')

In [ ]:
# 把MSE畫成折線圖

plt.figure(figsize=(8, 4))
plt.plot(length, mse_rnn, marker='o', label='SimpleRNN')
plt.axhline(BASELINE, color='gray', linestyle=':', label='Baseline') # 亂猜基準線
plt.xlabel('Sequence length')
plt.ylabel('Test MSE (lower is better)')
plt.title('Adding Problem: SimpleRNN')
plt.legend()
plt.ylim(-0.01, 0.2)
plt.show()

### LSTM:給記憶裝上閘門

<b>LSTM(長短期記憶網路,Long Short-Term Memory)</b>就是為了剛剛那個失敗而設計的。它在 RNN 的記憶上加了三道<b>閘門</b>,像一本可以主動決定「寫入什麼、擦掉什麼」的筆記本:

- <b>遺忘閘</b>:決定舊記憶哪些該丟掉(例如過時的行情已不重要)
- <b>輸入閘</b>:決定新資訊哪些值得寫進筆記
- <b>輸出閘</b>:決定此刻要根據筆記說什麼

關鍵在於:重要的資訊(例如剛剛 adding problem 裡「第一個標記位置的數字」)可以被閘門<b>保護起來</b>,一路傳到最後而不被中間的干擾洗掉,學習訊號也能沿著這條受保護的通道傳回開頭。它到底守不守得住?下面我們就用<b>完全相同</b>的記憶力測試換上 LSTM,把它的曲線疊到剛剛 SimpleRNN 那張圖上,直接比一比。

<b>GRU</b> 是 LSTM 的簡化版:少一道閘、參數更少,實務上常常一樣好用(課堂練習會讓你自己比較)。

從下一節開始,序列會變得更長:氣象案例一次要看 <b>120 個時間步</b>,正是 SimpleRNN 撐不住的長度。所以接下來的主角,都是 LSTM。


### 驗證:換上 LSTM,再測一次

說得再好聽,不如直接測。我們沿用<b>完全相同</b>的 adding problem(標記兩個位置、要模型相加)和相同的長度,只把循環層從 <code>SimpleRNN</code> 換成 <code>LSTM</code>,把它的誤差曲線疊到剛剛那張圖上——看閘門是不是真的守得住記憶。

In [ ]:
mse_lstm = []
for T in length:
  e = adding_test(T, layers.LSTM)
  mse_lstm.append(e)
  print(f'序列長度: {T}: LSTM MSE: {e:.4f}')

In [ ]:
# 把MSE畫成折線圖

plt.figure(figsize=(8, 4))
plt.plot(length, mse_rnn, marker='o', label='SimpleRNN')
plt.plot(length, mse_lstm, marker='s', label='LSTM')
plt.axhline(BASELINE, color='gray', linestyle=':', label='Baseline') # 亂猜基準線
plt.xlabel('Sequence length')
plt.ylabel('Test MSE (lower is better)')
plt.title('Adding Problem: SimpleRNN')
plt.legend()
plt.ylim(-0.01, 0.2)
plt.show()

<div style="background-color:#e0f2f1;border-left:5px solid #00897b;padding:12px 16px;margin:8px 0;border-radius:4px;color:#1a1a1a;"><b>🎯 對照結果:閘門真的有用</b><br>同一個題目、同樣的長度,SimpleRNN 到 100 步誤差已爬到 baseline(等於亂猜),LSTM 卻依然貼近 0、幾乎完美解出。差別只在把循環層從 <code style="background:rgba(0,0,0,0.07);color:#1a1a1a;padding:1px 5px;border-radius:3px;">SimpleRNN</code> 換成 <code style="background:rgba(0,0,0,0.07);color:#1a1a1a;padding:1px 5px;border-radius:3px;">LSTM</code>——閘門讓「第一個標記的數字」被保護著一路傳到第二個標記,學習訊號也能沿著這條受保護的通道傳回前面。這就是接下來(序列更長的真實資料)主角都是 LSTM 的原因。</div>

---
## 第 3 節　真實資料:用 LSTM 預測 12 小時後的氣溫

正弦波和記憶力測試證明了兩件事:工具沒問題,而且我們知道該選誰。這一節的視窗長達 <b>120 個時間步</b>,遠超過 SimpleRNN 的記憶極限,所以從這裡開始,主角都是 LSTM。

在往下走之前,先把本單元的地圖攤開來。U4 其實是一條<b>「訊號由強到弱」的階梯</b>,接下來的每一節就是往下走一階,看模型的表現如何隨訊號強度變化:

| 階 | 資料 | 訊號強度 | 位置 |
|:---:|---|---|---|
| 1 | 正弦波(合成) | 100%,自己造的 | 第 2~2.5 節(已完成) |
| 2 | **氣溫(真實)** | **強:物理規律,不會被「用掉」** | **第 3 節(本節)** |
| 3 | 波動率(**合成**) | 中等:群聚與均值回歸(生成時刻意寫入) | 第 5~6 節 |
| 4 | 報酬率方向(**合成**) | 幾乎為零:模擬效率市場 | 第 7 節 |

⚠️ <b>關於第 5~7 節的資料,要先誠實說明</b>:那三節用的是<b>合成</b>的報酬率資料(執行時程式會印出「使用合成報酬率資料」提示)。合成程式裡刻意寫入了波動率群聚與均值回歸,所以第 6 節那個漂亮的勝利,某種程度上是<b>被資料生成過程保證</b>的。它示範的是「當結構確實存在時,LSTM 抓不抓得到」,而不是「LSTM 在真實股市能賺錢」。換句話說:<b>本單元唯一在真實資料上取得的成功,就是這一節的氣溫。</b>

所以氣象案例不是繞路,它是這條階梯上不可跳過的一站:<b>真實資料的成功標尺</b>。正弦波只能證明程式沒寫錯,沒辦法證明 LSTM 應付得了真實世界的雜訊、缺值和不完美。氣溫資料可以:德國 Jena 氣象站 2009~2016 年的觀測紀錄,每 10 分鐘一筆,總共 8 年份,是不折不扣的真實資料。

而且氣象資料有一個金融資料沒有的好處:<b>它的規律是物理造成的。</b>白天升溫、晚上降溫、夏熱冬冷,這些規律不會因為「大家都知道」就消失。股市不一樣:任何一個好用的規律,只要被夠多人發現,就會被交易掉(這正是 U1~U3 一路碰壁的原因)。

所以這一節的定位是:<b>先看看「資料裡真的有訊號」的時候,LSTM 能強到什麼程度。</b>有了這把標尺,兩節之後回到金融資料、看到模型只小贏 baseline 時,你才分得出來那是「模型不行」還是「資料裡本來就沒那麼多東西」。

要問的問題是:<b>看過去 5 天的天氣,能不能猜出 12 小時後的氣溫?</b>


In [ ]:
import os
import pandas as pd
import zipfile

# 你本機的zip檔路徑
zip_path = 'jena_climate_2009_2016.csv.zip' # 壓縮檔的位置
extract_dir = 'jena_climate' # 要解壓到的資料夾

# 解壓縮
with zipfile.ZipFile(zip_path, 'r') as zf:
  zf.extractall(extract_dir)

# 讀取檔案
df_w = pd.read_csv('/content/jena_climate/jena_climate_2009_2016.csv')
df_w

# 降取樣:原始資料是每10分鐘一筆，一小時有6筆，所以我們改成每隔6筆取1筆資料
df_w = df_w.iloc[5::6].reset_index(drop=True)
df_w

# 取出三個欄位轉成float32的numpy陣列
temp = df_w['T (degC)'].to_numpy('float32') # 氣溫，這一節要預測的目標
pres = df_w['p (mbar)'].to_numpy('float32') # 氣壓，輔助特徵，因為和天氣變化有關
rhum = df_w['rh (%)'].to_numpy('float32') # 相對溼度，輔助特徵

# 簡單印出資料規模與氣溫範圍
print(f'每小時資料共 {len(temp)} 筆(約 {len(temp)/24/365:.1f} 年)')
print(f'氣溫範圍 {temp.min():.1f} ~ {temp.max():.1f} C, 平均 {temp.mean():.1f} C')


### 先看看資料長什麼樣

時序建模的第一步永遠是<b>把資料畫出來</b>。下面兩張圖分別用不同的時間尺度看同一份氣溫資料:拉遠看得到<b>季節</b>循環(一年一次),拉近則看得到<b>日夜</b>循環(一天一次)。

這兩個循環就是 LSTM 待會兒要學的東西——先用眼睛確認它們真的存在,免得訓練完不知道模型到底該學到什麼。


In [ ]:
# 設定兩種時間尺度:一整年(季節) vs 連續5天(日夜)

fig, axes = plt.subplots(2, 1, figsize=(12, 5))

axes[0].plot(temp[:24*365], linewidth=0.5)
axes[0].set_title('Temperature - firs year')
axes[0].set_ylabel('degC')

axes[1].plot(temp[:24*5], marker='o', markersize=3)
axes[1].set_title('Temperature - firs 5 days')
axes[1].set_ylabel('degC')
axes[1].set_xlabel('hours')

plt.tight_layout()
plt.show()

### 題目設定:用過去 5 天,預測 12 小時後

- <b>輸入 X</b>:過去 120 小時(5 天)的氣溫、氣壓、相對濕度
- <b>答案 y</b>:12 小時後那一刻的氣溫
- <b>對照 baseline</b>:naive persistence,「12 小時後就跟現在一樣」

先記住這個 baseline 的邏輯:<b>「未來 = 現在」</b>。它和金融理論裡的<b>隨機漫步假設</b>(明天的價格,最好的猜測就是今天的價格)是同一件事。這位對手在氣象這一站會被打得很慘,但兩節之後回到金融,它會變得非常難纏,先記住它的臉。

為什麼挑 12 小時?因為這是 naive baseline <b>最狼狽</b>的距離:如果現在是中午最熱的時候,12 小時後正好是半夜最冷的時候,「跟現在一樣」會錯得很離譜。而 LSTM 只要學會「一天有一次冷熱循環」,就能大幅超越它。

這個選擇本身就是教學重點:等一下的教師註解會告訴你,把 12 小時改成 24 小時,結論會出現有趣的翻轉。


In [ ]:
WIN = 120 # 看過去120小時的氣溫(5天)
HOR = 12 # 預測12小時後那一刻的氣溫

# 把三個特徵疊成一個2維的陣列(時間點, 3)
# axis=-1, 代表疊在最後一個維度:每個時間點從1個數字變成3個數字並排
# 原本temp/pres/rhum各是一長條，所以疊完變成每個小時會有3個觀測值的表格
feats = np.stack([temp, pres, rhum], axis=-1)

# 準備三個空的list, X_w用來存題目, y_w是用來存答案, naive_w用來存naive的預測
X_w, y_w, naive_w = [], [], []

# i會從WIN開始,然後到len(temp)- HOR + 1結束
for i in range(WIN, len(temp)- HOR + 1):
  # 題目:代表第i小時以前的三個特徵
  X_w.append(feats[i-WIN:i])
  # 答案:12小時後的氣溫
  y_w.append(temp[i + HOR - 1])
  # 直接拿第i-1小時的氣溫當作12小時後的氣溫
  naive_w.append(temp[i - 1])

X_w = np.array(X_w, 'float32')
y_w = np.array(y_w, 'float32')
naive_w = np.array(naive_w, 'float32')

# 切分訓練集和測試集
sp_w = int(len(X_w) * 0.8)
Xtr_w, Xte_w = X_w[:sp_w], X_w[sp_w:]
ytr_w, yte_w = y_w[:sp_w], y_w[sp_w:]
naive_te = naive_w[sp_w:]

# print(Xtr_w.shape)
# print(Xte_w.shape)
# print(ytr_w.shape)
# print(yte_w.shape)
# print(naive_te.shape)

# 資料標準化:把三個特徵各自轉成[平均和標準差]
# 為什麼要做標準差，因為氣溫-20~40，氣壓約1000以上, 濕度0~100%
mu_w, sd_w = Xtr_w.mean(axis=(0, 1)), Xtr_w.std(axis=(0, 1))
# print(mu_w)
# print(sd_w)
Xtr_ws = (Xtr_w - mu_w) / sd_w
Xte_ws = (Xte_w - mu_w) / sd_w

print(f'樣本{X_w.shape} (樣本數, 時間步, 特徵數)')
print(f'訓練 {len(Xtr_ws)} 筆|測試 {len(Xte_ws)} 筆')

In [ ]:
tf.random.set_seed(42)

weather_model = keras.Sequential([
          keras.Input(shape=(WIN, 3)),
          layers.LSTM(32),
          layers.Dense(1)
])

weather_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

weather_model.fit(Xtr_ws, ytr_w, validation_split=0.15,
          epochs=8, batch_size=128, verbose=1)

In [ ]:
pred_w = weather_model.predict(Xte_ws, verbose=0).ravel()

# MAE(平均絕對誤差):平均猜錯幾度
# 方法1:LSTM的預測誤差 = 預測溫度 - 真實溫度的平均
mae_lstm = np.abs(pred_w - yte_w).mean()
# 方法2:將前一格存好的溫度當作是此刻的溫度，直接拿來做預測，這是第2笨的基準
mae_naive = np.abs(naive_te - yte_w).mean() # baseline1:12小時後的溫度就是現在的溫度
# 方法3:常數預測，永遠都猜訓練集的平均溫，這是最笨的基準
mae_const = np.abs(ytr_w.mean() - yte_w).mean() # baseline2:永遠都猜訓練集的平均溫度

print(f'LSTM {mae_lstm:.2f}')
print(f'navie 12小時後 = 現在 {mae_naive:.2f}')
print(f'永遠猜平均溫度 {mae_const:.2f}')
print(f'LSTM 比 naive準 {(mae_naive - mae_lstm) / mae_naive * 100:.0f}%')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(yte_w[:120], label='Actual', linewidth=2)
plt.plot(pred_w[:120], label='LSTM', linestyle='--')
plt.plot(naive_te[:120], label='Naive(=now)', linestyle=':', alpha=0.8)
plt.title('Temperature +12h : Actual vs LSTM vs Naive')
plt.ylabel('degC')
plt.xlabel('hours')
plt.legend()
plt.show()

# 課堂專題｜用 LSTM 預測股票市場的「波動率」

**深度學習與金融應用｜Unit 4 專題**

這份專題是第 3 節「氣象案例」的**金融版**。你會用**一模一樣的建模流程**（多變數時序 → 滑動視窗 → LSTM → 和 baseline 誠實比較），只是把預測標的從「氣溫」換成「股票的波動率」。

> ⚠️ **為什麼預測波動率，不預測股價？**
> 氣溫有物理造成的、不會消失的規律，所以 LSTM 能大勝。但股價的「方向」接近隨機（效率市場），硬預測股價只會被 naive baseline 電爆。波動率不一樣——它有**群聚**（大波動會連在一起）和**均值回歸**（極端終會回落）的結構，是股市裡少數「有脾氣可循」的東西。這也是為什麼實務上選擇權定價、風險管理都在估波動率，而不是賭漲跌。

---

## 專題目標

用**過去 60 天**的市場資訊，預測**未來 5 天實際發生的波動率**，並和兩個 baseline 誠實比較：

- **輸入 X**：過去 60 天的 4 個特徵（已實現波動率、絕對報酬率、日內高低振幅、成交量）
- **答案 y**：未來 5 天報酬率的標準差（未來實際的波動程度）
- **對照 baseline**：naive persistence（「未來波動率 = 現在」）、常數（「永遠猜平均」）

---

## 金融三鐵則檢查（動手前先確認）

- **鐵則一（用報酬率不用價格）**：所有特徵都建立在報酬率或波動率上，不直接餵股價。
- **鐵則二（時序切分不 shuffle）**：前 80% 訓練、後 20% 測試，絕不打亂。
- **鐵則三（嚴禁前視）**：每個特徵都只用「當天及以前」的資訊；答案（未來波動率）絕不能洩漏進特徵。

---

## 任務一：下載資料與特徵工程

從 Yahoo Finance 下載國巨（2327.TW）的日資料，計算 4 個特徵。

**要求：**
1. 下載 `2327.TW` 從 2008 年至今的 OHLCV 日資料。
2. 計算對數報酬率。
3. 完成 `realized_vol()`：回傳「過去 w 天報酬率的標準差」。
4. 做出 4 個特徵並確認長度一致：
   - `rv`：已實現波動率（過去 10 天）—— 主訊號
   - `abs_ret`：絕對報酬率 —— 當天波動大小
   - `hl_range`：`log(High/Low)` —— 日內振幅
   - `log_vol`：`log(成交量+1)` —— 市場活躍度



---

## 任務二：滑動視窗、切分與標準化

把長序列切成一筆一筆的監督式樣本。

**要求：**
1. 設 `WIN = 60`、`HORIZON = 5`。
2. 用 `np.stack(..., axis=-1)` 把 4 個特徵疊成 `(時間點, 4)` 的多變數時序。
3. 用滑動視窗做出 `X`（過去 60 天 × 4 特徵）、`y`（未來 5 天報酬率的標準差）、`naive`（當前波動率）。
4. 時序切分（前 80% / 後 20%，**不 shuffle**）。
5. 標準化：`mean`/`std` **只從訓練集**算，沿 `axis=(0,1)`，四個特徵各自標準化。



---

## 任務三：建模、訓練與三方比較

蓋一個和氣象案例相同的 LSTM，訓練後和兩個 baseline 比 MAE。

**要求：**
1. 建模：`Input(60,4)` → `LSTM(32)` → `Dense(1)`。
2. **想清楚**：輸出層 `Dense(1)` 要不要加激活函數？為什麼？（提示：這是回歸還是分類？）
3. 編譯：`loss="mse"`、`metrics=["mae"]`。想一想這兩者分工是什麼？
4. 訓練後，計算並列出三種方法的 MAE：LSTM、naive、常數。
5. 算出 LSTM 相對 naive 的百分比差距。

---

## 任務四：誠實解讀（最重要的一題）

跑完之後，對照氣象案例的結果，回答：

1. 氣象案例裡 LSTM 贏 naive 約 **五成**。你的股票波動率專題，LSTM 贏了多少？（或是**輸了**？）
2. 為什麼同樣的模型、同樣是真實資料，兩者結果差這麼多？
3. 如果 LSTM 贏不過 naive persistence，這代表「LSTM 沒用」嗎？還是代表別的意思？
4. 換一個隨機種子（`tf.random.set_seed`）再跑一次，結論會變嗎？這說明了什麼？


In [ ]:
import yfinance as yf
# 把2327下載下來
raw = yf.download('2327.TW', start='2008-01-01', auto_adjust=True, progress=False)
close = raw['Close'].to_numpy('float64').ravel()
high = raw['High'].to_numpy('float64').ravel()
low = raw['Low'].to_numpy('float64').ravel()
volume = raw['Volume'].to_numpy('float64').ravel()

# 對數報酬率
log_ret = np.zeros(len(close))
log_ret[1:] = np.diff(np.log(close))

# 已實現的波動率:過去W天報酬率的標準差

ret_series = pd.Series(log_ret)
rolling_window = ret_series.rolling(10, min_periods=1)
rolling_std = rolling_window.std(ddof=0)
rv = rolling_std.to_numpy()

# print(f'已實現波度率長度 {len(rv)}')
# print(f'前5個值 {rv[:5]}')

# 計算出4個特徵
rv = rv # 已實現波動率
abs_ret = np.abs(log_ret) # 當天波動的大小
hl_range = np.log(high/ low) # 日內高低振幅
log_vol = np.log(volume + 1) # 市場活耀度

# ---------- 任務二:滑動視窗、切分與標準化 ----------
WIN = 60      # 看過去 60 天
HORIZON = 5   # 預測未來 5 天的波動率

feats = np.stack([rv, abs_ret, hl_range, log_vol], axis=-1)   # (時間點, 4)

X_v, y_v, naive_v = [], [], []
for i in range(WIN, len(rv) - HORIZON):
    X_v.append(feats[i-WIN:i])                 # 題目:過去 60 天、4 特徵(不含 i)
    y_v.append(log_ret[i:i+HORIZON].std())     # 答案:未來 5 天實際波動率
    naive_v.append(rv[i-1])                    # naive:未來 = 現在的波動率
X_v = np.array(X_v, "float32")
y_v = np.array(y_v, "float32")
naive_v = np.array(naive_v, "float32")

# 時序切分,不 shuffle
sp = int(len(X_v) * 0.8)
Xtr, Xte = X_v[:sp], X_v[sp:]
ytr, yte = y_v[:sp], y_v[sp:]
naive_te = naive_v[sp:]

# 標準化:統計量只從訓練集算,四個特徵各自標準化(沿 axis=(0,1))
mu, sd = Xtr.mean(axis=(0, 1)), Xtr.std(axis=(0, 1))
Xtr_s = (Xtr - mu) / sd
Xte_s = (Xte - mu) / sd

print(f"樣本 shape: {X_v.shape} (樣本數, 時間步, 特徵數)")
print(f"訓練 {len(Xtr)}｜測試 {len(Xte)}")


# ---------- 任務三:建模、訓練與三方比較 ----------
tf.random.set_seed(42)
vol_model = keras.Sequential([
    keras.Input(shape=(WIN, 4)),   # 60 個時間步、每步 4 個特徵
    layers.LSTM(32),
    layers.Dense(1),               # 回歸:預測波動率數值,不加激活函數
])
vol_model.compile(optimizer="adam", loss="mse", metrics=["mae"])
vol_model.fit(Xtr_s, ytr, validation_split=0.15, epochs=20, batch_size=64, verbose=0)

pred = vol_model.predict(Xte_s, verbose=0).ravel()

mae_lstm  = np.abs(pred - yte).mean()
mae_naive = np.abs(naive_te - yte).mean()      # baseline①:未來 = 現在
mae_const = np.abs(ytr.mean() - yte).mean()    # baseline②:永遠猜平均

print(f"{'方法':<26}{'MAE'}")
print(f"{'LSTM':<28}{mae_lstm:.5f}")
print(f"{'naive(未來=現在)':<24}{mae_naive:.5f}")
print(f"{'常數(永遠猜平均)':<24}{mae_const:.5f}")
print(f"\n→ LSTM 相對 naive:{(mae_naive - mae_lstm)/mae_naive*100:+.0f}%")

In [ ]:
import matplotlib.pyplot as plt

# 常數 baseline 的預測值(一條水平線)
const_pred = ytr.mean()

# 畫測試集最前面 120 天,看四條線的差別
N = 120
plt.figure(figsize=(12, 4))
plt.plot(yte[:N],       label="Actual",            linewidth=2)
plt.plot(pred[:N],      label="LSTM",              linestyle="--")
plt.plot(naive_te[:N],  label="Naive (= now)",     linestyle=":", alpha=0.8)
plt.axhline(const_pred, label="Const (= mean)",    color="gray", linestyle="-.", alpha=0.7)
plt.title("Volatility +5d: Actual vs LSTM vs Naive vs Const (first 120 days of test set)")
plt.ylabel("realized volatility"); plt.xlabel("days"); plt.legend()
plt.tight_layout(); plt.show()